# PREPARATION DU DATASET

##### Import des librairies :

In [47]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from sklearn.pipeline import make_pipeline

##### Chargement des données d'entrainement et de test brutes :

In [48]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")

In [49]:
print(data_train.isna().sum())

id                  0
Age                 0
BodyweightKg        0
MeetYear            0
Sex                 0
Equipment           0
AgeClass            0
BirthYearClass      0
WeightClassKg       0
Tested              0
Country             0
State               0
Federation          0
ParentFederation    0
MeetCountry         0
MeetState           0
Sanctioned          0
TotalKg             0
dtype: int64


Il n'y a aucune valeur manquante (NaN) dans le dataset d'entraînement

In [50]:
print(data_test.isna().sum())

id                  0
Age                 0
BodyweightKg        0
MeetYear            0
Sex                 0
Equipment           0
AgeClass            0
BirthYearClass      0
WeightClassKg       0
Tested              0
Country             0
State               0
Federation          0
ParentFederation    0
MeetCountry         0
MeetState           0
Sanctioned          0
dtype: int64


Il y a 9 valeurs manquantes (NaN) dans le dataset de test pour la variable `WeightClassKg`

### Variables continues :

Rien ne change pour ces variables ci : `Age`, `BodyweightKg`, `Meet_year`

### Variables catégorielles :

Il y a des variables ordinales, pour lesquelles l'ordre est important : `AgeClass`, `BirthYearClass`, `WeightClassKg`

Il y a des variables nominales (pas forcément textuelles !), pour lesquelles l'ordre ne compte pas : `Sex`, `Equipment`, `Tested`, `Country`,  `State`, `Federation`, `ParentFederation`, `MeetCountry`, `MeetState`, `Sanctioned`

Ces deux types de variables ne se traitent pas avec le même encodeur. On sépare les deux groupes :

##### Variables catégorielles ordinales :

In [51]:
print(data_train["AgeClass"].unique())

['16-17' '24-34' '50-54' '20-23' '18-19' '55-59' '40-44' '35-39' '60-64'
 '13-15' '45-49' '5-12' '65-69' '70-74' '80-84' 'Unknown' '75-79' '85-89'
 '90-999']


In [52]:
ordre_AgeClass = {"16-17": 16, "24-34": 29, "50-54": 52, "20-23": 21, "18-19": 18,
             "55-59": 57, "40-44": 42, "35-39": 37, "60-64": 62, 
             "13-15": 14, "45-49": 47, "5-12": 9, "65-69": 67, 
             "70-74": 72, "80-84": 82, "Unknown": -1, "75-79": 77,
             "85-89": 87, "90-999": 95}

data_train["AgeClass"] = data_train["AgeClass"].map(ordre_AgeClass)
data_test["AgeClass"] = data_test["AgeClass"].map(ordre_AgeClass)

print(data_train["AgeClass"].unique()) # vérification

[16 29 52 21 18 57 42 37 62 14 47  9 67 72 82 -1 77 87 95]


In [53]:
print(data_train["BirthYearClass"].unique())

['14-18' '24-39' '50-59' '19-23' '40-49' 'Unknown' '60-69' '70-999']


In [54]:
ordre_BirthYearClass = {"14-18": 16, "24-39": 31, "50-59": 54, "19-23": 21, "40-49": 44, "Unknown": -1,
              "60-69": 64, "70-999": 70}

data_train["BirthYearClass"] = data_train["BirthYearClass"].map(ordre_BirthYearClass)
data_test["BirthYearClass"] = data_test["BirthYearClass"].map(ordre_BirthYearClass)

print(data_train["BirthYearClass"].unique()) # vérification

[16 31 54 21 44 -1 64 70]


In [55]:
print(data_train["WeightClassKg"].unique())

['66' '90' '84+' '52' '125' '82.5' '140' '140+' '93' '75' '110' 'Unknown'
 '69' '100+' '57' '105' '67.3' '120+' '63' '100' '120' '47' '84' '67.5'
 '110+' '60' '48' '74' '59' '83' '82.1' '51.9' '76' '124.7' '56' '117.7+'
 '53' '72' '109.7' '75+' '59.8' '89.8' '139.7+' '125+' '43' '90+' '72.5'
 '64' '67.5+' '99.7' '74.8' '70' '67.1' '82.3' '+' '56+' '85' '44.2' '65'
 '40' '51.7' '50.5' '55.7' '94' '44' '118' '77' '117.7' '69+' '85+' '47.8'
 '58.5' '93+' '105+' '60.1' '139.7' '90.7+' '143' '127' '58' '80' '124.7+'
 '49' '35' '95' '30' '63+' '145+' '33' '56.5' '90.7' '77.5' '55.5' '136'
 '36' '136.5' '81.5' '145' '63.5' '83+' '46' '81.5+' '102' '113.5' '62'
 '74+' '47.5' '155' '60+' '-93' '81.6' '43.5' '82.5+' '143+' '120.2+'
 '91.1' '148' '76+' '56.7' '50' '72+' '44.5' '34.9' '31' '115' '80+' '57+'
 '78.9' '83.9' '70.3' '54' '113.4' '79.5' '155+' '93.8' '36.2' '47.6'
 '135.9' '54.4' '104.7' '109' '73' '109+' '55' '117.5' '120.2']


In [56]:
ordre_WeightClassKg = {
    # --- Inconnus / parasites ---
    "Unknown": -1, "+": -1,

    # --- Valeurs numériques ---
    "30": 30, "31": 31, "33": 33, "34.9": 34.9, "35": 35, "36": 36, "36.2": 36.2,
    "40": 40, "43": 43, "43.5": 43.5, "44": 44, "44.2": 44.2, "44.5": 44.5,
    "46": 46, "47": 47, "47.5": 47.5, "47.6": 47.6, "47.8": 47.8, "48": 48,
    "49": 49, "50": 50, "50.5": 50.5, "51.7": 51.7, "51.9": 51.9, "52": 52,
    "53": 53, "54": 54, "54.4": 54.4, "55": 55, "55.5": 55.5, "55.7": 55.7,
    "56": 56, "56.5": 56.5, "56.7": 56.7, "57": 57, "58": 58, "58.5": 58.5,
    "59": 59, "59.8": 59.8, "60": 60, "60.1": 60.1, "62": 62, "63": 63,
    "63.5": 63.5, "64": 64, "65": 65, "66": 66, "67.1": 67.1, "67.3": 67.3,
    "67.5": 67.5, "69": 69, "70": 70, "70.3": 70.3, "72": 72, "72.5": 72.5,
    "73": 73, "74": 74, "74.8": 74.8, "75": 75, "76": 76, "77": 77,
    "77.5": 77.5, "78.9": 78.9, "79.5": 79.5, "80": 80, "81.5": 81.5,
    "81.6": 81.6, "82.1": 82.1, "82.3": 82.3, "82.5": 82.5, "83": 83,
    "83.9": 83.9, "84": 84, "85": 85, "89.8": 89.8, "90": 90, "90.7": 90.7,
    "91.1": 91.1, "93": 93, "-93": 93, "93.8": 93.8, "94": 94, "95": 95,
    "99.7": 99.7, "100": 100, "102": 102, "104.7": 104.7, "105": 105,
    "109": 109, "109.7": 109.7, "110": 110, "113.4": 113.4, "113.5": 113.5,
    "115": 115, "117.5": 117.5, "117.7": 117.7, "118": 118, "120": 120,
    "120.2": 120.2, "124.7": 124.7, "125": 125, "127": 127, "135.9": 135.9,
    "136": 136, "136.5": 136.5, "139.7": 139.7, "140": 140, "143": 143,
    "145": 145, "148": 148, "155": 155,

    # --- Catégories "X+" (juste au-dessus de X) ---
    "56+": 56.01, "57+": 57.01, "60+": 60.01, "63+": 63.01, "67.5+": 67.51,
    "69+": 69.01, "72+": 72.01, "74+": 74.01, "75+": 75.01, "76+": 76.01,
    "80+": 80.01, "81.5+": 81.51, "82.5+": 82.51, "83+": 83.01, "84+": 84.01,
    "85+": 85.01, "90+": 90.01, "90.7+": 90.71, "93+": 93.01, "100+": 100.01,
    "105+": 105.01, "109+": 109.01, "110+": 110.01, "117.7+": 117.71,
    "120+": 120.01, "120.2+": 120.21, "124.7+": 124.71, "125+": 125.01,
    "139.7+": 139.71, "140+": 140.01, "143+": 143.01, "145+": 145.01,
    "155+": 155.01,
}

data_train["WeightClassKg"] = data_train["WeightClassKg"].map(ordre_WeightClassKg)
data_test["WeightClassKg"] = data_test["WeightClassKg"].map(ordre_WeightClassKg)

print(data_train["WeightClassKg"].unique()) # vérification

[ 66.    90.    84.01  52.   125.    82.5  140.   140.01  93.    75.
 110.    -1.    69.   100.01  57.   105.    67.3  120.01  63.   100.
 120.    47.    84.    67.5  110.01  60.    48.    74.    59.    83.
  82.1   51.9   76.   124.7   56.   117.71  53.    72.   109.7   75.01
  59.8   89.8  139.71 125.01  43.    90.01  72.5   64.    67.51  99.7
  74.8   70.    67.1   82.3   56.01  85.    44.2   65.    40.    51.7
  50.5   55.7   94.    44.   118.    77.   117.7   69.01  85.01  47.8
  58.5   93.01 105.01  60.1  139.7   90.71 143.   127.    58.    80.
 124.71  49.    35.    95.    30.    63.01 145.01  33.    56.5   90.7
  77.5   55.5  136.    36.   136.5   81.5  145.    63.5   83.01  46.
  81.51 102.   113.5   62.    74.01  47.5  155.    60.01  81.6   43.5
  82.51 143.01 120.21  91.1  148.    76.01  56.7   50.    72.01  44.5
  34.9   31.   115.    80.01  57.01  78.9   83.9   70.3   54.   113.4
  79.5  155.01  93.8   36.2   47.6  135.9   54.4  104.7  109.    73.
 109.01  55.   117.5  120

In [57]:
print(data_test["WeightClassKg"].unique()) # vérification

[110.   100.    -1.    82.5   60.    75.    84.    59.8   74.8   90.01
  69.    84.01  66.    57.   105.    93.   120.    90.    67.5   83.
  47.    74.   140.01  44.2   82.1   59.   140.    52.   120.01  63.
 125.    56.    72.    76.    53.    30.   110.01  67.1   89.8   44.
 117.71  67.3   99.7   64.   125.01  43.    55.7  100.01  51.7   77.
  48.   124.7   60.1  109.7   85.    85.01  70.    94.    58.    95.
  47.8  145.    65.    51.9  118.   124.71  93.01  80.    82.3  117.7
  40.   139.71  31.    75.01  55.5   50.5   62.    82.51  67.51 105.01
 102.    90.7   77.5  139.7     nan 113.5   83.01  76.01  35.    47.5
  56.01  54.    69.01 127.    58.5   70.3   74.01  57.01  63.01 143.01
  36.    49.    90.71 120.2   78.9  115.    81.6  104.7   72.01]


On doit remplir les valeurs manquantes pour la variable `WeightClassKg` (avec la médiane du dataset de train) :

In [58]:
data_test["WeightClassKg"] = data_test["WeightClassKg"].fillna(data_train["WeightClassKg"].median()) 

In [59]:
cols_nom = [
    "Sex", "Equipment",
    "Tested", "Country",
    "State", "Federation", "ParentFederation", "MeetCountry",
    "MeetState", "Sanctioned"]

# Ordre alphabétique par défaut de OrdinalEncoder
enc_nom = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-2) # je mets un code (-2) pour les NaN au passage
data_train[cols_nom] = enc_nom.fit_transform(data_train[cols_nom])
data_test[cols_nom] = enc_nom.transform(data_test[cols_nom])

In [60]:
print(data_train["Sex"].unique()) # vérification

[1. 0. 2.]


In [61]:
print(data_train.dtypes.value_counts())  # doit n'afficher que int64 / float64

float64    14
int64       4
Name: count, dtype: int64


### Vérifications :

In [62]:
print(data_train.isna().sum())

id                  0
Age                 0
BodyweightKg        0
MeetYear            0
Sex                 0
Equipment           0
AgeClass            0
BirthYearClass      0
WeightClassKg       0
Tested              0
Country             0
State               0
Federation          0
ParentFederation    0
MeetCountry         0
MeetState           0
Sanctioned          0
TotalKg             0
dtype: int64


In [63]:
print(data_test.isna().sum())

id                  0
Age                 0
BodyweightKg        0
MeetYear            0
Sex                 0
Equipment           0
AgeClass            0
BirthYearClass      0
WeightClassKg       0
Tested              0
Country             0
State               0
Federation          0
ParentFederation    0
MeetCountry         0
MeetState           0
Sanctioned          0
dtype: int64


##### On enregistre ces nouveaux datasets pour éviter de relancer tote la préparation à chaque fois :

In [64]:
data_train.to_csv("data_train_clean.csv", index=True)
data_test.to_csv("data_test_clean.csv", index=True)